# 05 — Train & Evaluate
Huber GBRT in log10 space with isotonic calibration.

- Strict embedding loaders (order-preserving, pad/truncate, zero-fill).
- Fixed tabular builder (uses only split subset rows).
- **tmdb_id** de-duplication policy to ensure a unique index before assembly.
- Split ID lists de-duplicated (order-preserving).

In [ ]:
import os, json, joblib, numpy as np, pandas as pd
from pathlib import Path

ART_DIR   = Path(os.getenv("ART_DIR",   "~/boxoffice/artifacts")).expanduser()
DATA_DIR  = Path(os.getenv("DATA_DIR",  "~/boxoffice/data")).expanduser()
IMG_DIR   = Path(os.getenv("IMG_DIR",   "~/boxoffice/data/img")).expanduser()

TEXT_DIR  = Path(os.getenv("TEXT_DIR",  ART_DIR / "text_emb")).expanduser()
CLIP_DIR  = Path(os.getenv("CLIP_DIR",  ART_DIR / "clip_emb")).expanduser()
ROI_DIR   = Path(os.getenv("ROI_DIR",   ART_DIR / "roi_feat")).expanduser()

def load_json(path): return json.loads(Path(path).read_text(encoding="utf-8"))

# Handy helper
def _vec(path, dim):
    try:
        v = np.load(path); v = np.asarray(v, np.float32).ravel()
        if   v.size == dim: return v
        elif v.size >  dim: return v[:dim]
        else:               return np.pad(v, (0, dim - v.size))
    except Exception:
        return np.zeros(dim, np.float32)


In [15]:
## FOR RELOAD ONLY
from sklearn.isotonic import IsotonicRegression
from sklearn.ensemble import GradientBoostingRegressor

# Model + calibrator
gb  = joblib.load(ART_DIR / "model_gbr_huber.joblib")
iso = joblib.load(ART_DIR / "calibrator_isotonic.joblib")

# Saved predictions (create y/p variables if later cells expect them)
pred_train = pd.read_csv(ART_DIR / "pred_train.csv")
pred_val   = pd.read_csv(ART_DIR / "pred_val.csv")
pred_test  = pd.read_csv(ART_DIR / "pred_test.csv")
pred_covid = pd.read_csv(ART_DIR / "pred_covid.csv") if (ART_DIR/"pred_covid.csv").exists() else None

# (Optional) populate the names used in your notebook
ytr,  p_tr,  p_tr_c  = pred_train["y_log10"].values, pred_train["pred_raw"].values, pred_train["pred_cal"].values
yva,  p_va,  p_va_c  = pred_val["y_log10"].values,   pred_val["pred_raw"].values,   pred_val["pred_cal"].values
yte,  p_te,  p_te_c  = pred_test["y_log10"].values,  pred_test["pred_raw"].values,  pred_test["pred_cal"].values
if pred_covid is not None:
    ycv, p_cv, p_cv_c = pred_covid["y_log10"].values, pred_covid["pred_raw"].values, pred_covid["pred_cal"].values

results_main = pd.read_csv(ART_DIR / "results_train_eval.csv") if (ART_DIR/"results_train_eval.csv").exists() else None

In [1]:
import os as _os
_os.environ["TQDM_NOTEBOOK"] = "0"
from tqdm import tqdm
try: tqdm.monitor_interval = 0
except Exception: pass

In [2]:
# Imports & params
import os, json
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

MASTER_CSV = os.getenv("MASTER_CSV", os.path.expanduser("~/boxoffice/data/movies_master_preprocessed.csv"))
ART_DIR    = Path(os.getenv("ART_DIR",    os.path.expanduser("~/boxoffice/artifacts")))
TEXT_DIR   = Path(os.getenv("TEXT_DIR",   str(ART_DIR / "text_emb")))
CLIP_DIR   = Path(os.getenv("CLIP_DIR",   str(ART_DIR / "clip_emb")))
ROI_DIR    = Path(os.getenv("ROI_DIR",    str(ART_DIR / "roi_feat")))

USE_TAB  = os.getenv("USE_TAB","1")=="1"
USE_TEXT = os.getenv("USE_TEXT","1")=="1"
USE_CLIP = os.getenv("USE_CLIP","1")=="1"
USE_ROI  = os.getenv("USE_ROI","1")=="1"
SEED = int(os.getenv("SEED","42"))

print("MASTER_CSV:", MASTER_CSV)
print("ART_DIR:", ART_DIR)
print("TEXT_DIR:", TEXT_DIR)
print("CLIP_DIR:", CLIP_DIR)
print("ROI_DIR:", ROI_DIR)
print("Features:", dict(USE_TAB=USE_TAB, USE_TEXT=USE_TEXT, USE_CLIP=USE_CLIP, USE_ROI=USE_ROI))

MASTER_CSV: C:\Users\Vaishob/boxoffice/data/movies_master_preprocessed.csv
ART_DIR: C:\Users\Vaishob\boxoffice\artifacts
TEXT_DIR: C:\Users\Vaishob\boxoffice\artifacts\text_emb
CLIP_DIR: C:\Users\Vaishob\boxoffice\artifacts\clip_emb
ROI_DIR: C:\Users\Vaishob\boxoffice\artifacts\roi_feat
Features: {'USE_TAB': True, 'USE_TEXT': True, 'USE_CLIP': True, 'USE_ROI': True}


In [3]:
# Load data & split IDs (dedup each split)
df = pd.read_csv(MASTER_CSV)
if "tmdb_id" not in df.columns:
    raise ValueError("master CSV must contain 'tmdb_id' column")
if "revenue_usd_2024" not in df.columns:
    raise ValueError("master CSV must contain 'revenue_usd_2024' column")

def load_ids(name):
    p = ART_DIR / f"{name}.csv"
    if not p.exists():
        return []
    s = pd.read_csv(p)["tmdb_id"].dropna().astype("Int64").astype(int)
    seen=set(); out=[]
    for v in s:
        v=int(v)
        if v not in seen:
            seen.add(v); out.append(v)
    return out

ids_train = load_ids("train_ids")
ids_val   = load_ids("val_ids")
ids_covid = load_ids("covid_diag_ids")
ids_test  = load_ids("test_ids")

print("split sizes:", len(ids_train), len(ids_val), len(ids_covid), len(ids_test))

split sizes: 3215 166 237 465


In [4]:
# --- Ensure unique tmdb_id index (de-dup dataframe + confirm policy) ---
dup_counts = df["tmdb_id"].value_counts()
dups = dup_counts[dup_counts > 1]
print(f"Duplicate tmdb_id rows: {int(dups.shape[0])}")
if not dups.empty:
    print("Top dup examples:", dups.head(10).to_dict())

# Dedup policy: keep first occurrence (stable). You may switch to other policies if desired.
df_u = df.drop_duplicates(subset=["tmdb_id"], keep="first").copy()

# Unique index for assembly
df_u_idx = df_u.set_index("tmdb_id")

def dedup_ids(lst):
    seen=set(); out=[]
    for v in lst:
        v=int(v)
        if v not in seen:
            seen.add(v); out.append(v)
    return out

ids_train = dedup_ids(ids_train)
ids_val   = dedup_ids(ids_val)
ids_covid = dedup_ids(ids_covid)
ids_test  = dedup_ids(ids_test)
print("After ID de-dup:", {"train": len(ids_train), "val": len(ids_val), "covid": len(ids_covid), "test": len(ids_test)})

Duplicate tmdb_id rows: 310
Top dup examples: {18221.0: 6, 672.0: 3, 10200.0: 2, 36123.0: 2, 8055.0: 2, 46705.0: 2, 11499.0: 2, 1487.0: 2, 14359.0: 2, 11321.0: 2}
After ID de-dup: {'train': 3215, 'val': 166, 'covid': 237, 'test': 465}


In [5]:
# Build vocabularies (from full df once), strict tabular builder
NUM_COLS = [
    "budget_usd_2024","runtime","popularity","vote_average","vote_count",
    "wiki_pv_61d_log1p","reddit_posts_log1p","reddit_comments_log1p","gt_raw_sum"
]

TOPK_GENRES = 15
if "genres_norm" in df_u.columns:
    gcounts = {}
    for s in df_u["genres_norm"].fillna("").astype(str):
        for g in [x for x in s.split(";") if x]:
            gcounts[g] = gcounts.get(g,0) + 1
    GENRES = [g for g,_ in sorted(gcounts.items(), key=lambda kv: kv[1], reverse=True)[:TOPK_GENRES]]
else:
    GENRES = []

if "distributor_bucket" in df_u.columns:
    DIST_UNIQ = sorted(df_u["distributor_bucket"].fillna("other").astype(str).unique().tolist())
else:
    DIST_UNIQ = []

def build_tabular_strict(sub: pd.DataFrame) -> np.ndarray:
    n = len(sub)
    # numeric
    num_blocks = []
    for c in NUM_COLS:
        col = sub[c] if c in sub.columns else pd.Series(np.nan, index=sub.index)
        num_blocks.append(col.astype(float).fillna(0.0).to_numpy().reshape(n,1))
    Xnum = np.hstack(num_blocks) if num_blocks else np.zeros((n,0), dtype=np.float32)

    # genres
    if GENRES and "genres_norm" in sub.columns:
        G = np.zeros((n, len(GENRES)), dtype=np.float32)
        for i, s in enumerate(sub["genres_norm"].fillna("").astype(str)):
            toks = set([x for x in s.split(";") if x])
            for j,g in enumerate(GENRES):
                if g in toks: G[i,j]=1.0
        Xg = G
    else:
        Xg = np.zeros((n,0), dtype=np.float32)

    # distributor
    if DIST_UNIQ and "distributor_bucket" in sub.columns:
        cats = sub["distributor_bucket"].fillna("other").astype(str)
        map_idx = {u:i for i,u in enumerate(DIST_UNIQ)}
        Xd = np.zeros((n, len(DIST_UNIQ)), dtype=np.float32)
        for i,s in enumerate(cats):
            j = map_idx.get(s, None)
            if j is not None: Xd[i,j] = 1.0
    else:
        Xd = np.zeros((n,0), dtype=np.float32)

    X = np.hstack([Xnum, Xg, Xd]).astype(np.float32, copy=False)
    if X.shape[0] != n:
        raise ValueError(f"build_tabular_strict produced wrong row count: {X.shape} vs n={n}")
    return X

In [6]:
# Strict loaders for embeddings (order-preserving, padding/truncation)
from pathlib import Path
import numpy as np

def _as_1d(x):
    x = np.asarray(x)
    if x.ndim == 1: return x
    if x.ndim == 2 and 1 in x.shape: return x.reshape(-1)
    raise ValueError(f"Expected 1-D vector per id, got shape {x.shape}")

def load_vecs_strict(dir_path: Path, ids, name="feat"):
    dir_path = Path(dir_path)
    d = None; n_missing = n_malformed = n_fixdim = 0
    # detect common dimension
    for tid in ids:
        fp = dir_path / f"{tid}.npy"
        if fp.exists():
            try:
                v = _as_1d(np.load(fp, allow_pickle=False))
                d = int(v.shape[0]); break
            except Exception:
                continue
    if d is None:
        return np.zeros((len(ids),0), dtype=np.float32), 0, len(ids), 0, 0

    rows = []
    for tid in ids:
        fp = dir_path / f"{tid}.npy"
        if not fp.exists():
            n_missing += 1
            rows.append(np.zeros((d,), dtype=np.float32))
            continue
        try:
            v = _as_1d(np.load(fp, allow_pickle=False)).astype(np.float32, copy=False)
        except Exception:
            n_malformed += 1
            rows.append(np.zeros((d,), dtype=np.float32))
            continue
        if v.shape[0] == d:
            rows.append(v)
        elif v.shape[0] > d:
            rows.append(v[:d]); n_fixdim += 1
        else:
            pad = np.zeros((d,), dtype=np.float32)
            pad[:v.shape[0]] = v
            rows.append(pad); n_fixdim += 1
    X = np.vstack(rows).astype(np.float32, copy=False)
    print(f"[{name}] shape={X.shape}, d={d}, missing={n_missing}, malformed={n_malformed}, fixed_dim={n_fixdim}")
    return X, d, n_missing, n_malformed, n_fixdim

In [7]:
# Assemble per split (STRICT) using df_u_idx (unique index)
df_idx = df_u_idx  # << use the de-duplicated, unique-index dataframe

def assemble(ids):
    ids = [int(i) for i in ids if int(i) in df_idx.index]
    sub = df_idx.loc[ids].copy()

    blocks = []; names = []

    if USE_TAB:
        X_tab = build_tabular_strict(sub)
        print(f"[tab ] shape={X_tab.shape}")
        blocks.append(X_tab); names.append("tabular")

    if USE_TEXT:
        Xt, _, _, _, _ = load_vecs_strict(Path(TEXT_DIR), ids, name="text")
        blocks.append(Xt); names.append("text")

    if USE_CLIP:
        Xc, _, _, _, _ = load_vecs_strict(Path(CLIP_DIR), ids, name="clip")
        blocks.append(Xc); names.append("clip")

    if USE_ROI:
        Xr, _, _, _, _ = load_vecs_strict(Path(ROI_DIR), ids, name="roi")
        blocks.append(Xr); names.append("roi")

    n = len(ids)
    for nm, Xb in zip(names, blocks):
        if Xb.shape[0] != n:
            raise ValueError(f"Row mismatch in block '{nm}': {Xb.shape[0]} vs expected {n}")

    X = np.hstack(blocks) if blocks else np.zeros((n,0), dtype=np.float32)
    y = np.log10(np.maximum(sub['revenue_usd_2024'].astype(float).values, 1.0))
    return np.array(ids, dtype=int), X, y

train_ids, Xtr, ytr = assemble(ids_train)
val_ids,   Xva, yva = assemble(ids_val)
cov_ids,   Xcv, ycv = assemble(ids_covid)
test_ids,  Xte, yte = assemble(ids_test)
print("shapes:", Xtr.shape, Xva.shape, Xcv.shape, Xte.shape)

[tab ] shape=(3215, 55)
[text] shape=(3215, 768), d=768, missing=14, malformed=0, fixed_dim=0
[clip] shape=(3215, 768), d=768, missing=276, malformed=0, fixed_dim=0
[roi] shape=(3215, 4096), d=4096, missing=276, malformed=0, fixed_dim=0
[tab ] shape=(166, 55)
[text] shape=(166, 768), d=768, missing=1, malformed=0, fixed_dim=0
[clip] shape=(166, 768), d=768, missing=32, malformed=0, fixed_dim=0
[roi] shape=(166, 4096), d=4096, missing=32, malformed=0, fixed_dim=0
[tab ] shape=(237, 55)
[text] shape=(237, 768), d=768, missing=0, malformed=0, fixed_dim=0
[clip] shape=(237, 768), d=768, missing=38, malformed=0, fixed_dim=0
[roi] shape=(237, 4096), d=4096, missing=38, malformed=0, fixed_dim=0
[tab ] shape=(465, 55)
[text] shape=(465, 768), d=768, missing=0, malformed=0, fixed_dim=0
[clip] shape=(465, 768), d=768, missing=47, malformed=0, fixed_dim=0
[roi] shape=(465, 4096), d=4096, missing=47, malformed=0, fixed_dim=0
shapes: (3215, 5687) (166, 5687) (237, 5687) (465, 5687)


In [8]:
# Train GBRT (Huber) + isotonic calibration
params = dict(n_estimators=2000, learning_rate=0.02, max_depth=3, min_samples_leaf=20,
              subsample=1.0, random_state=SEED, loss="huber", alpha=0.9)
gb = GradientBoostingRegressor(**params).fit(Xtr, ytr)

p_tr = gb.predict(Xtr)
p_va = gb.predict(Xva)
p_cv = gb.predict(Xcv)
p_te = gb.predict(Xte)

iso = IsotonicRegression(out_of_bounds="clip").fit(p_va, yva)
p_tr_c = iso.transform(p_tr)
p_va_c = iso.transform(p_va)
p_cv_c = iso.transform(p_cv)
p_te_c = iso.transform(p_te)
print("trained + calibrated")

trained + calibrated


In [9]:
# Metrics + save
def smape(y_true_lin, y_pred_lin, eps=1e-9):
    den = (np.abs(y_true_lin) + np.abs(y_pred_lin)) + eps
    return 200.0 * np.mean(np.abs(y_pred_lin - y_true_lin) / den)

def metrics(y_log, p_log):
    y = np.power(10.0, y_log); p = np.power(10.0, p_log)
    return {
        "MAE_lin": float(mean_absolute_error(y, p)),
        "SMAPE": float(smape(y, p)),
        "R2_log10": float(r2_score(y_log, p_log)),
        "MAE_log10": float(mean_absolute_error(y_log, p_log)),
    }

rows = []
for name, yt, pt in [
    ("train_raw", ytr, p_tr),
    ("val_raw",   yva, p_va),
    ("covid_raw", ycv, p_cv),
    ("test_raw",  yte, p_te),
    ("train", ytr, p_tr_c),
    ("val",   yva, p_va_c),
    ("covid", ycv, p_cv_c),
    ("test",  yte, p_te_c),
]:
    m = metrics(yt, pt); m["split"] = name; rows.append(m)

res = pd.DataFrame(rows).set_index("split")
display(res)

ART_DIR.mkdir(parents=True, exist_ok=True)
res.to_csv(ART_DIR / "results_train_eval.csv", index=True)
print("saved metrics ->", ART_DIR / "results_train_eval.csv")

joblib.dump(gb,  ART_DIR / "model_gbr_huber.joblib")
joblib.dump(iso, ART_DIR / "calibrator_isotonic.joblib")
print("saved model + calibrator")

,MAE_lin,SMAPE,R2_log10,MAE_log10
split,,,,
train_raw,2.558749e+07,19.162450,0.951446,0.085961
val_raw,7.264531e+07,57.869938,0.710123,0.292427
covid_raw,5.436452e+07,88.764962,0.482805,0.491391
test_raw,6.124760e+07,73.259762,0.650585,0.372519
train,3.527194e+07,25.058175,0.931911,0.112414
val,5.538034e+07,50.665343,0.769747,0.250896
covid,6.132501e+07,82.165448,0.571958,0.441458
test,5.987300e+07,69.151066,0.676862,0.350362


saved metrics -> C:\Users\Vaishob\boxoffice\artifacts\results_train_eval.csv
saved model + calibrator


In [24]:
# assume gbr (GradientBoostingRegressor) and iso (IsotonicRegression) exist
joblib.dump(gbr, ART_DIR / "model_gbr_huber.joblib")
joblib.dump(iso, ART_DIR / "calibrator_isotonic.joblib")

# Save per-split predictions (raw & calibrated) for reproducibility
pd.DataFrame({
    "tmdb_id": train_ids,
    "y_log10": ytr,
    "pred_raw": p_tr,
    "pred_cal": p_tr_c
}).to_csv(ART_DIR / "pred_train.csv", index=False)

pd.DataFrame({
    "tmdb_id": val_ids,
    "y_log10": yva,
    "pred_raw": p_va,
    "pred_cal": p_va_c
}).to_csv(ART_DIR / "pred_val.csv", index=False)

# COVID split is optional; save only if variables exist
if "covid_diag_ids" in globals() and "ycv" in globals() and "p_cv" in globals() and "p_cv_c" in globals():
    pd.DataFrame({
        "tmdb_id": covid_diag_ids,
        "y_log10": ycv,
        "pred_raw": p_cv,
        "pred_cal": p_cv_c
    }).to_csv(ART_DIR / "pred_covid.csv", index=False)

pd.DataFrame({
    "tmdb_id": test_ids,
    "y_log10": yte,
    "pred_raw": p_te,
    "pred_cal": p_te_c
}).to_csv(ART_DIR / "pred_test.csv", index=False)

# Save config snapshot
save_json({
    "model": "GBR(huber)",
    "params": getattr(gb, "get_params", lambda: {})(),
    "calibrator": "IsotonicRegression",
    "blocks_used": {"text": True, "clip": True, "roi": True, "tabular": True}
}, ART_DIR / "05_training_meta.json")

In [25]:
# === SAVE COVID PREDICTIONS (uses your variable names) ===
import os, numpy as np, pandas as pd
from pathlib import Path

ART_DIR = Path(os.getenv("ART_DIR", "~/boxoffice/artifacts")).expanduser()
ART_DIR.mkdir(parents=True, exist_ok=True)

def _to_1d(a):
    a = np.asarray(a)
    return a.reshape(-1) if a.ndim > 1 else a

def _have(names):
    missing = [n for n in names if n not in globals()]
    return len(missing) == 0, missing

# Prefer in-memory; fallback to CSV if ids aren’t in memory
def _covid_ids():
    if "covid_diag_ids" in globals():
        try:
            # Series/DataFrame/array/list all supported
            if hasattr(covid_diag_ids, "to_frame") or hasattr(covid_diag_ids, "values"):
                return _to_1d(getattr(covid_diag_ids, "values", covid_diag_ids)).astype(int).tolist()
            return _to_1d(covid_diag_ids).astype(int).tolist()
        except Exception:
            pass
    csv_path = ART_DIR / "covid_diag_ids.csv"
    if csv_path.exists():
        return pd.read_csv(csv_path)["tmdb_id"].astype(int).tolist()
    return []

ok, missing = _have(["ycv", "p_cv", "p_cv_c"])
ids = _covid_ids()

if not ok:
    print("[covid save] Skipped: missing variables ->", missing)
elif len(ids) == 0:
    print("[covid save] Skipped: no covid_diag_ids found (in memory or ART_DIR/covid_diag_ids.csv).")
else:
    y  = _to_1d(ycv).astype(float)
    pr = _to_1d(p_cv).astype(float)
    pc = _to_1d(p_cv_c).astype(float)
    n  = min(len(ids), len(y), len(pr), len(pc))
    if n == 0:
        print("[covid save] Skipped: zero-length after alignment.")
    else:
        df = pd.DataFrame({
            "tmdb_id": np.asarray(ids[:n], dtype=int),
            "y_log10": y[:n],
            "pred_raw": pr[:n],
            "pred_cal": pc[:n],
        })
        out = ART_DIR / "pred_covid.csv"
        df.to_csv(out, index=False)
        print(f"[covid save] Wrote {n} rows -> {out}")


[covid save] Wrote 237 rows -> C:\Users\Vaishob\boxoffice\artifacts\pred_covid.csv


In [26]:
list(ART_DIR.glob("*"))

[WindowsPath('C:/Users/Vaishob/boxoffice/artifacts/01_checkpoint.json'),
 WindowsPath('C:/Users/Vaishob/boxoffice/artifacts/05_training_meta.json'),
 WindowsPath('C:/Users/Vaishob/boxoffice/artifacts/abl_clip_only.json'),
 WindowsPath('C:/Users/Vaishob/boxoffice/artifacts/abl_clip_roi.json'),
 WindowsPath('C:/Users/Vaishob/boxoffice/artifacts/abl_roi_only.json'),
 WindowsPath('C:/Users/Vaishob/boxoffice/artifacts/abl_tab_clip.json'),
 WindowsPath('C:/Users/Vaishob/boxoffice/artifacts/abl_tab_clip_roi.json'),
 WindowsPath('C:/Users/Vaishob/boxoffice/artifacts/abl_tab_only.json'),
 WindowsPath('C:/Users/Vaishob/boxoffice/artifacts/abl_tab_roi.json'),
 WindowsPath('C:/Users/Vaishob/boxoffice/artifacts/abl_tab_text.json'),
 WindowsPath('C:/Users/Vaishob/boxoffice/artifacts/abl_tab_text_clip.json'),
 WindowsPath('C:/Users/Vaishob/boxoffice/artifacts/abl_tab_text_clip_roi.json'),
 WindowsPath('C:/Users/Vaishob/boxoffice/artifacts/abl_tab_text_roi.json'),
 WindowsPath('C:/Users/Vaishob/boxoff

In [27]:
for p in [TEXT_DIR, CLIP_DIR, ROI_DIR]:
    print(p, "→", len(list(p.glob("*.npy"))), "files")

C:\Users\Vaishob\boxoffice\artifacts\text_emb → 4476 files
C:\Users\Vaishob\boxoffice\artifacts\clip_emb → 4177 files
C:\Users\Vaishob\boxoffice\artifacts\roi_feat → 4177 files


In [28]:
import shutil
shutil.make_archive("boxoffice_artifacts_backup", "zip", ART_DIR)

'C:\\Users\\Vaishob\\boxoffice\\notebooks\\boxoffice_artifacts_backup.zip'